In [1]:
from typing import TypedDict

class State(TypedDict):
    messages: list

In [2]:
def node_a_fn(state: State):
    print(f"[node_a] 입력 메시지: {state['messages']}")
    return {"messages": ["node_a 처리 완료"]}
    
def node_b_fn(state: State):
    print(f"[node_b] 입력 메시지: {state['messages']}")
    return {"messages": ["node_b 처리 완료"]}

In [3]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# StateGraph 정의
graph_builder = StateGraph(State)

graph_builder.add_node("node_a", node_a_fn)
graph_builder.add_node("node_b", node_b_fn)

graph_builder.add_edge(START, "node_a")
graph_builder.add_edge("node_a", "node_b")
graph_builder.add_edge("node_b", END)

# 체크포인터와 함께 컴파일
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [4]:

# 그래프 실행 - 각 노드 실행 후 자동으로 체크포인트 생성
config = {"configurable": {"thread_id": "1"}}
result = graph.invoke({"messages": ["안녕하세요"]}, config=config)

[node_a] 입력 메시지: ['안녕하세요']
[node_b] 입력 메시지: ['node_a 처리 완료']


체크포인트 히스토리 조회

In [5]:
# 전체 실행 히스토리 조회 (최신순)
history = list(graph.get_state_history(config))

In [6]:
history

[StateSnapshot(values={'messages': ['node_b 처리 완료']}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f10df04-5018-60d8-8002-3e4996532e3b'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-02-20T00:08:22.623458+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f10df04-5015-696a-8001-058cd68512f0'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'messages': ['node_a 처리 완료']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f10df04-5015-696a-8001-058cd68512f0'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-02-20T00:08:22.622449+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f10df04-5012-6efb-8000-b524dbf55104'}}, tasks=(PregelTask(id='5e9237fd-3767-645e-01ae-ea1865f44208', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interru

In [7]:
print(f"총 {len(history)}개의 체크포인트:")
for i, snapshot in enumerate(history):
    print(f"{i}. Step {snapshot.metadata.get('step', 'N/A')}")
    print(f"   Checkpoint ID: {snapshot.config['configurable']['checkpoint_id']}")
    print(f"   Next nodes: {snapshot.next}")

총 4개의 체크포인트:
0. Step 2
   Checkpoint ID: 1f10df04-5018-60d8-8002-3e4996532e3b
   Next nodes: ()
1. Step 1
   Checkpoint ID: 1f10df04-5015-696a-8001-058cd68512f0
   Next nodes: ('node_b',)
2. Step 0
   Checkpoint ID: 1f10df04-5012-6efb-8000-b524dbf55104
   Next nodes: ('node_a',)
3. Step -1
   Checkpoint ID: 1f10df04-5011-6b16-bfff-90a9686b5e53
   Next nodes: ('__start__',)


상태 수정

In [8]:
# 특정 체크포인트 선택 (next 필드로 안전하게 찾기)
target_snapshot = None
for snapshot in history:
    if snapshot.next == ("node_b",):  # node_b 실행 직전 체크포인트
        target_snapshot = snapshot
        break

In [9]:
target_snapshot

StateSnapshot(values={'messages': ['node_a 처리 완료']}, next=('node_b',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f10df04-5015-696a-8001-058cd68512f0'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-02-20T00:08:22.622449+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f10df04-5012-6efb-8000-b524dbf55104'}}, tasks=(PregelTask(id='5e9237fd-3767-645e-01ae-ea1865f44208', name='node_b', path=('__pregel_pull', 'node_b'), error=None, interrupts=(), state=None, result={'messages': ['node_b 처리 완료']}),), interrupts=())

In [10]:
# 상태 수정
graph.update_state(
    target_snapshot.config,
    {"messages": ["수정된 메시지"]},
    as_node="node_a"  # node_a가 업데이트한 것으로 처리 → 다음은 node_b
)

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f10df17-0585-646a-8002-e9f96cdc0c73'}}

체크포인트에서 실행 재개

In [11]:
# 선택한 체크포인트의 설정으로 재실행
resume_config = target_snapshot.config

In [12]:
resume_config

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f10df04-5015-696a-8001-058cd68512f0'}}

In [13]:
# None을 입력으로 전달 - 저장된 상태에서 계속 실행 (새 fork 생성)
result = graph.invoke(None, config=resume_config)

[node_b] 입력 메시지: ['node_a 처리 완료']


In [14]:
result

{'messages': ['node_b 처리 완료']}